In [ ]:
import datetime
import pandas as pd
import numpy as np

In [ ]:
print(datetime.datetime.now())

2026-06-04 14:19:34.187494


In [ ]:
!which python

/usr/local/bin/python


In [ ]:
!python --version

Python 3.12.13


In [ ]:
!echo $PYTHONPATH

/env/python


# Credit Risk Classification

This project develops machine learning workflow for credit risk classification using applicant demographic, financial and credit-history information.

This evaluates:

1. Baseline classification

2. Feature Engineering

3. Feature Selection

4. Hyperparameter tuning

5. Final evaluation on unseen test data

Macro F1 used as primary evaluation metric to evaluate performance.

Key Result: - The final tuned Random Forest model achieved a Macro F1 score of 0.7845 on unseen dataset.

Technologies: - Python, Pandas, NumPy, scikit-learn, Random Forest, Feature Engineering, Feature Selection, GridSearchCV, Cross-Validation.

## 1.0: Load data and split

In [ ]:


# First, we'll read the provided labeled training data
df3 = pd.read_csv("https://drive.google.com/uc?export=download&id=1wOhyCnvGeY4jplxI8lZ-bbYN3zLtickf")
df3.info()

from sklearn.model_selection import train_test_split

X = df3.drop('BadCredit', axis=1)
y = df3['BadCredit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   UserID             6000 non-null   object
 1   Sex                6000 non-null   object
 2   PreviousDefault    6000 non-null   int64 
 3   FirstName          6000 non-null   object
 4   LastName           6000 non-null   object
 5   NumberPets         6000 non-null   int64 
 6   PreviousAccounts   6000 non-null   int64 
 7   ResidenceDuration  6000 non-null   int64 
 8   Street             6000 non-null   object
 9   LicensePlate       6000 non-null   object
 10  BadCredit          6000 non-null   int64 
 11  Amount             6000 non-null   int64 
 12  Married            6000 non-null   int64 
 13  Duration           6000 non-null   int64 
 14  City               6000 non-null   object
 15  Purpose            6000 non-null   object
 16  DateOfBirth        6000 non-null   object


## 1.1: Baseline model

A fixed random_state value was used throughout the analysis to ensure that the results are reproducible and can be consistently replicated.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import cross_val_score

In [ ]:
# Inspecting the training data
print("Shape of X_train:", X_train.shape)
print("First few rows:\n", X_train.head(3))
print("Missing values per column:\n", X_train.isnull().sum())
print("Number of duplicate rows:", X_train.duplicated().sum())
print("Class distribution in y_train:\n", y_train.value_counts())


Shape of X_train: (4800, 16)
First few rows:
            UserID Sex  PreviousDefault FirstName LastName  NumberPets  \
3897  236-22-6766   M                0     Jerry    Black           2   
5628  766-20-5986   F                0     Julia    Jones           0   
1756  744-25-5747   F                0   Abigail  Estrada           2   

      PreviousAccounts  ResidenceDuration                Street LicensePlate  \
3897                 0                  2       0466 Brown Wall      3-U8282   
5628                 2                  2  6095 Larson Causeway      LWO 912   
1756                 0                  3    293 Michael Divide      715 OQT   

      Amount  Married  Duration            City    Purpose DateOfBirth  
3897    3329        0        12  New Roberttown  Household  1970-04-22  
5628    2996        0        36       Ericmouth  Household  1964-06-19  
1756    2470        0        24       East Jill     NewCar  1975-02-17  
Missing values per column:
 UserID              

In [ ]:
# Data Cleaning
drop_cols = ['UserID', 'FirstName', 'LastName', 'Street', 'LicensePlate']
X_train_clean = X_train.drop(columns=drop_cols, errors='ignore')

The above variables were dropped because they are highly unique identifiers and are unlikely to help in creating meaningful credit risk patterns. Retaining such variables could create unwanted noise.

In [ ]:
# Encoding of categorical and numerical features
categorical_cols = X_train_clean.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train_clean.select_dtypes(exclude=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['Sex', 'City', 'Purpose', 'DateOfBirth']
Numerical columns: ['PreviousDefault', 'NumberPets', 'PreviousAccounts', 'ResidenceDuration', 'Amount', 'Married', 'Duration']


In [ ]:
# Preprocessing piplines
from sklearn.pipeline import Pipeline

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy= 'most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown= 'ignore'))
])
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy= 'mean'))
])

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_cols),
    ('num', numerical_transformer, numerical_cols)
])

In [ ]:
# Pipeline
from sklearn.ensemble import RandomForestClassifier

baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [ ]:
# Cross Validation
from sklearn.model_selection import cross_val_score

scores = cross_val_score(baseline_pipeline, X_train_clean, y_train,
                         cv=5, scoring= 'f1_macro')
print("Macro F1 scores for each fold:", scores)
print("Mean Macro F1 score:", scores.mean())

Macro F1 scores for each fold: [0.52238219 0.49731801 0.53823954 0.51702163 0.53417522]
Mean Macro F1 score: 0.5218273193268614


The baseline Random Forest model achieved a Mean Macro F1 score of approximately 0.52 using 5 fold cross validation.
This low score indicates that the model is not capturing important patterns in the data. Further improvements will be discussed in the following sections.

## 1.2: Adding feature engineering

In [ ]:
# Creating copy of training data
X_train_fe = X_train_clean.copy()

In [ ]:
# Creating New Features
X_train_fe['AmountPerMonth'] =(
    X_train_fe['Amount'] /
    (X_train_fe['Duration'] +1)

)

In [ ]:
X_train_fe['FinancialHistory'] = (
    X_train_fe['PreviousAccounts'] +
    X_train_fe['PreviousDefault']
)

In [ ]:
# Converting DateOfBirth into datetime
X_train_fe['DateOfBirth'] = pd.to_datetime(X_train_fe['DateOfBirth'])

# Creating Age Feature
X_train_fe ['Age'] = 2025 - X_train_fe['DateOfBirth'].dt.year

Age was determined by referencing a fixed year to allow for the consistent comparison of all the applicants by age at the time when data was collected.

In [ ]:
# since age created
X_train_fe = X_train_fe.drop(columns=['DateOfBirth'])

In [ ]:
categorical_cols_fe = (
    X_train_fe.select_dtypes(include=['object']).columns.tolist()
)
numerical_cols_fe = (
    X_train_fe.select_dtypes(exclude=['object']).columns.tolist()
)

In [ ]:
print(categorical_cols_fe)
print(numerical_cols_fe)

['Sex', 'City', 'Purpose']
['PreviousDefault', 'NumberPets', 'PreviousAccounts', 'ResidenceDuration', 'Amount', 'Married', 'Duration', 'AmountPerMonth', 'FinancialHistory', 'Age']


In [ ]:
# Preprocessing pipeline
categorical_transformer_fe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy= 'most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown= 'ignore'))
])

numerical_transformer_fe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

In [ ]:
preprocessor_fe = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer_fe,
         categorical_cols_fe),
        ('num', numerical_transformer_fe,
         numerical_cols_fe)

    ]
)

In [ ]:
fe_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('classifier',
     RandomForestClassifier(random_state=42))
])

In [ ]:
fe_scores = cross_val_score(
    fe_pipeline, X_train_fe, y_train, cv=5, scoring= 'f1_macro'
)

print("Feature Engineering Macro F1 Scores;")
print(fe_scores)

print("\n Mean Macro F1 Score:")
print(fe_scores.mean())

Feature Engineering Macro F1 Scores;
[0.77222038 0.69023141 0.77685182 0.78189896 0.77437007]

 Mean Macro F1 Score:
0.7591145273434243


In [ ]:
print("Baseline Mean MacroF1:", scores.mean())
print("Feature Engineering Mean Macro F1:", fe_scores.mean())

Baseline Mean MacroF1: 0.5218273193268614
Feature Engineering Mean Macro F1: 0.7591145273434243


The Baseline model achieved a mean Macro F1 score of 0.52, whereas the feature engineered model achieved approximately 0.759. There is a huge improvement in the performance. The new created features (age, amount per month, and financial history) provided additional information and thus, Random Forest model was able to discriminate better between good and bad credit applicants.

## 1.3: Adding feature selection

In [ ]:
# importing tools
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

In [ ]:
# creating new pipeline
fs_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('feature_selection',
     SelectKBest(
         score_func=f_classif,
         k=20

     )),
    ('classifier',
     RandomForestClassifier(
         random_state=42
     ))
])

In [ ]:
# Evaluating cross val score
fs_scores = cross_val_score(
    fs_pipeline, X_train_fe, y_train, cv=5, scoring='f1_macro'
)
print("Feature Selection Macro F1 Score:")
print(fs_scores)

print("\nMean Macro F1 Score:")
print(fs_scores.mean())

Feature Selection Macro F1 Score:
[0.77287864 0.73816039 0.80482204 0.78410453 0.76170442]

Mean Macro F1 Score:
0.7723340057647763


In [ ]:
import pandas as pd
results = pd.DataFrame({
    'Stage': [
        'Baseline', 'Feature Engineering', 'Feature Selection'
    ],
    'Mean Macro F1':[
        scores.mean(), fe_scores.mean(), fs_scores.mean()
    ]
})
print(results)

                 Stage  Mean Macro F1
0             Baseline       0.521827
1  Feature Engineering       0.759115
2    Feature Selection       0.772334


After applying feature selection in Section 1.3, the scores increased from 0.759 to 0.772. This indicates that removing less informative features helped the Random Forest model to focus on the most relevant features and improved overall performance.

## 1.4: Adding hyperparameter tuning

In [ ]:
# importing GridSearchCV

from sklearn.model_selection import GridSearchCV

In [ ]:
# Creating Tuning Pipeline
tuning_pipeline = Pipeline([
    ('preprocessor', preprocessor_fe),
    ('feature_selection',
     SelectKBest(
         score_func=f_classif,
         k=20
     )),
    ('classifier',
     RandomForestClassifier(
         random_state=42
     ))
])

In [ ]:
# Defining Parameter Grid
param_grid ={
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth':[
        None, 10, 20
    ],
    'classifier__min_samples_split':[
        2, 5, 10
    ]
}

In [ ]:
# Creating GridSearchCV
grid_search = GridSearchCV(
    estimator = tuning_pipeline,
    param_grid = param_grid,
    cv=5,
    scoring= 'f1_macro',
    n_jobs=-1
)

In [ ]:
_ = grid_search.fit(X_train_fe, y_train)

In [ ]:
# Best Model
print("Best Model:")
print(grid_search.best_estimator_)

print("\nBest Macro F1 Score;")
print(grid_search.best_score_)

Best Model:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'City', 'Purpose']),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer())]),
                                                  ['PreviousDefault',
                                                   'NumberPets',
                                                   'PreviousAccounts',
                

In [ ]:
# Comparing the results
print("Baseline Mean Macro F1:",
      scores.mean())
print("Feature Engineering Mean Macro F1:",
      fe_scores.mean())
print("Feature Selection Mean Macro F1:",
      fs_scores.mean())
print("Hyperparameter Tuned Mean Macro F1:",
      grid_search.best_score_)

Baseline Mean Macro F1: 0.5218273193268614
Feature Engineering Mean Macro F1: 0.7591145273434243
Feature Selection Mean Macro F1: 0.7723340057647763
Hyperparameter Tuned Mean Macro F1: 0.775762848063477


After applying hyperparameter tuning using GridSearchCV, the score went up slightly to approximately 0.776. Thus, greater performance gains were achieved through feature engineering and feature selection, while hyperparameter tuning provided a minor additional improvement.

Hyperparameter tuning produced the highest Macro F1 score (0.776). The tuned Random Forest model will be used in the final testing evaluation stage.

## 1.5: Performance estimation on testing data

In [ ]:
# Best Model
best_model = grid_search.best_estimator_

In [ ]:
# Testing data preparation
X_test_fe = X_test.copy()

In [ ]:
# Adding Features
# FINANCIAL FEATURE -
X_test_fe['FinancialHistory'] = (
    X_test_fe['PreviousAccounts']+
    X_test_fe['PreviousDefault']
)

# AMOUNT PER MONTH FEATURE -
X_test_fe['AmountPerMonth'] = (
    X_test_fe['Amount']/
    (X_test_fe['Duration']+1)
)

# AGE FEATURE -
X_test_fe['DateOfBirth'] = pd.to_datetime(
    X_test_fe['DateOfBirth']
)
X_test_fe['Age'] = (
    2025 - X_test_fe['DateOfBirth'].dt.year
)

In [ ]:
# Dropping columns
X_test_fe = X_test_fe.drop(
    columns = ['UserID', 'FirstName', 'LastName', 'Street', 'LicensePlate', 'DateOfBirth'],
    errors='ignore'
)

In [ ]:
# Generating predictions
y_pred = best_model.predict(
    X_test_fe
)

In [ ]:
# Importing the metrics
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix
)

In [ ]:
# Final Macro F1 score
test_macro_f1 = f1_score(
    y_test, y_pred, average = 'macro'
)
print(
    "Test Macro F1 Score:",
    test_macro_f1
)

Test Macro F1 Score: 0.7844659155879679


In [ ]:
# CSV and F1
print(
    "Best Cross Validation Macro F1:",
    grid_search.best_score_
)

print(
    "Test Macro F1:",
    test_macro_f1
)

Best Cross Validation Macro F1: 0.775762848063477
Test Macro F1: 0.7844659155879679


The final tuned Random Forest model was evaluated on the unknown testing dataset. The model achieved Macro F1 score of 0.7844 on test data, which is slightly higher than the best cross - validation score obtained during hyperparameter tuning (0.776).
The similarity between these two suggests that the model works well on unseen data and is not overfitting.
Since the model performed uniformly on both training validation process and independent test dataset, it is logical to expect that the model would perform alike on future loan applications.

CONCLUSION:

This project developed an end-to-end machine learning workflow for credit risk classification. Starting from baseline Macro F1 score of 0.5218, performance improved through feature engineering, feature selection, and hyperparameter tuning.

The final tuned Random Forest model achieved a Macro F1 score of 0.7845.

The project demonstrates how structured preprocessing, feature development, feature selection, model tuning, and independent evaluation can be combined to build a reproducible machine learning workflow for credit risk classification.